#  NSMC 감정 분류 Fine-tuning Project
### klue/bert-base → Naver Sentiment Movie Corpus

---

##  프로젝트 목표 

| # | 학습목표 | 평가기준 | 구현 위치 |
|---|---------|---------|----------|
| 1 | 모델·데이터 정상 로드 및 작동 확인 | klue/bert-base를 NSMC로 fine-tuning, 정상 동작 확인 | STEP 1~3 |
| 2 | Preprocessing 개선 + Fine-tuning으로 성능 개선 | **Validation accuracy ≥ 90%** | STEP 4 |
| 3 | Bucketing 성공 적용 + 결과 비교분석 | 연산 속도 vs 모델 성능 **trade-off** 분석 | STEP 5 |

##  전체 흐름

```
STEP 1: NSMC 로드/분석 → Huggingface Dataset 구성
STEP 2: klue/bert-base + tokenizer 불러오기
STEP 3: 전처리 → baseline 학습 (정상 동작 확인)
STEP 4: 전처리·하이퍼파라미터 개선 → 90%+ 달성
STEP 5: Bucketing (group_by_length) 적용 → STEP 4와 시간/성능 비교
```

##  핵심 직관

> 지난 mini BERT pretrain이 **"한국어 일반 표현 학습"** 이었다면,  
> 이번 fine-tuning은 그 일반 표현을 **"긍정/부정 분류"라는 specific task**에 맞추는 단계.  
> 우리가 직접 학습시켰던 작은 BERT 대신, KLUE 팀이 대규모로 학습시킨 `klue/bert-base`를 명장으로 영입\.
>
>  
> 홍명보 ㅂㅂ

In [3]:
# Colab 환경이면 먼저 설치 (Jupyter local이면 생략 가능)
# !pip install -q transformers datasets accelerate

import tensorflow
import numpy
import transformers
import datasets

print(f"tensorflow   : {tensorflow.__version__}")
print(f"numpy        : {numpy.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")

I0000 00:00:1779336809.151423     710 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779336810.698302     710 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


tensorflow   : 2.21.0
numpy        : 2.2.6
transformers : 5.9.0
datasets     : 4.8.5


In [2]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 18.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 6.3 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 2.7 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 74.4 MB/s eta 0:00:00ta 0:00:01
  Attempting uninstall: protobuf━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/16 [termcolor]
    Found existing installation: protobuf 5.29.3━━━━━━━━━━━━━━  4/16 [termcolor]
    Uninstalling protobuf-5.29.3:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/16 [termcolor]
      Successfully uninstalled protobuf-5.29.3━━━━━━━━━━━━━━━━  4/16 [termcolor]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [tensorflow]6 [tensorflow]


In [4]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 59.8 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 20.0.0
    Uninstalling pyarrow-20.0.0:
      Successfully uninstalled pyarrow-20.0.0━━━━━━━━━━━━━━━━━━━━━  1/12 [pyarrow]
  Attempting uninstall: dill0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/12 [pyarrow]
    Found existing installation: dill 0.4.0━━━━━━━━━━━━━━━━━━━━━━━  5/12 [dill]
    Uninstalling dill-0.4.0:m╸━━━━━━━━━━━━━━━━━━━━━━━  5/12 [dill]
      Successfully uninstalled dill-0.4.0━━━━━━━━━━━━━━━━━━━━━  5/12 [dill]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [datasets]/12 [datasets]


In [1]:
import pyarrow, datasets, transformers
print("pyarrow     :", pyarrow.__version__)      # 15.x 이상이면 OK
print("datasets    :", datasets.__version__)
print("transformers:", transformers.__version__)

pyarrow     : 24.0.0
datasets    : 4.8.5
transformers: 5.9.0


In [2]:
import tensorflow
import numpy
import transformers
import datasets

print(f"tensorflow   : {tensorflow.__version__}")
print(f"numpy        : {numpy.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")

I0000 00:00:1779334296.287580     386 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779334296.353498     386 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779334298.063111     386 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


tensorflow   : 2.21.0
numpy        : 2.2.6
transformers : 5.9.0
datasets     : 4.8.5


In [4]:
import pandas as pd
from datasets import Dataset, DatasetDict

URL_TRAIN = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
URL_TEST  = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"

# TSV (탭 구분) 파일. 컬럼: id, document, label
train_df = pd.read_csv(URL_TRAIN, sep="\t")
test_df  = pd.read_csv(URL_TEST,  sep="\t")

print(f"train: {len(train_df):,}  /  test: {len(test_df):,}")
print(train_df.head(3))

# pandas → Huggingface Dataset
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "test":  Dataset.from_pandas(test_df,  preserve_index=False),
})

print(raw_datasets)

train: 150,000  /  test: 50,000
         id                           document  label
0   9976970                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                  너무재밓었다그래서보는것을추천한다      0
DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})


In [5]:
import numpy as np

train_df = raw_datasets["train"].to_pandas()

# ① 라벨 분포
print("[라벨 비율]")
print(train_df["label"].value_counts(normalize=True))

# ② 결측/빈 문자열
n_null  = train_df["document"].isnull().sum()
n_empty = (train_df["document"].astype(str).str.strip() == "").sum()
print(f"\n[결측] null={n_null}, empty={n_empty}")

# ③ 문자 길이 분포
lengths = train_df["document"].astype(str).str.len()
print(f"\n[길이] mean={lengths.mean():.1f} | "
      f"median={lengths.median():.0f} | "
      f"95%={np.percentile(lengths, 95):.0f} | "
      f"max={lengths.max()}")

[라벨 비율]
label
0    0.501153
1    0.498847
Name: proportion, dtype: float64

[결측] null=5, empty=0

[길이] mean=35.2 | median=27 | 95%=107 | max=146


##  데이터 얼굴 보기 — 3가지 지표

> 모델 만지기 전에 **데이터부터 들여다보기**.  
> 여기서 나온 숫자가 STEP 3·5의 의사결정 근거가 됨.

| # | 보는 것 | 왜? | 어디서 쓰임 |
|---|---------|-----|-----------|
| ① | **라벨 비율** | 한쪽 쏠림 있으면 accuracy 의미 X | STEP 3 평가지표 |
| ② | **결측 / 빈 문자열** | 있으면 tokenizer가 에러 뱉음 | STEP 3 전처리 필터 |
| ③ | **문장 길이 분포** | `max_length` 정하는 근거 | STEP 3 토큰화, STEP 5 bucketing |

특히 ③의 **max와 median 차이가 크다 = bucketing 효과가 크다** → STEP 5 결과 미리 예측 가능.

In [6]:
import numpy as np

# Huggingface Dataset → pandas DataFrame (분석은 pandas가 편함)
train_df = raw_datasets["train"].to_pandas()

# ─────────────────────────────────────────────
# ① 라벨 분포: 50:50에 가까운지 확인
# ─────────────────────────────────────────────
print("[① 라벨 비율]")
print(train_df["label"].value_counts(normalize=True).rename({0: "부정", 1: "긍정"}))

# ─────────────────────────────────────────────
# ② 결측 / 빈 문자열 체크
#    → 있으면 STEP 3에서 .filter() 한 줄로 제거 예정
# ─────────────────────────────────────────────
n_null  = train_df["document"].isnull().sum()                       # NaN 개수
n_empty = (train_df["document"].astype(str).str.strip() == "").sum()  # 공백뿐인 문서
print(f"\n[② 결측] null={n_null}, empty={n_empty}")

# ─────────────────────────────────────────────
# ③ 문자 길이 분포 (글자 수 기준 — 토큰 수의 대용치)
#    → 95%값으로 max_length를 결정하는 게 정석
# ─────────────────────────────────────────────
lengths = train_df["document"].astype(str).str.len()
print(f"\n[③ 문자 길이]")
print(f"  mean   = {lengths.mean():.1f}")
print(f"  median = {lengths.median():.0f}")
print(f"  90%    = {np.percentile(lengths, 90):.0f}")
print(f"  95%    = {np.percentile(lengths, 95):.0f}")   # ← max_length 후보
print(f"  99%    = {np.percentile(lengths, 99):.0f}")
print(f"  max    = {lengths.max()}")                     # ← median과의 격차가 bucketing 효과

[① 라벨 비율]
label
부정    0.501153
긍정    0.498847
Name: proportion, dtype: float64

[② 결측] null=5, empty=0

[③ 문자 길이]
  mean   = 35.2
  median = 27
  90%    = 75
  95%    = 107
  99%    = 139
  max    = 146


##  결과 해석 방법

코드 실행 후 나오는 숫자에서 **다음 두 결정**을 내림:

### 결정 1. `max_length` 선택
- **95% 값이 50자 안쪽** → `max_length = 64` (토큰 ≈ 글자 × 1.2~1.5)
- **95% 값이 70~90자** → `max_length = 96`
- **95% 값이 100자 이상** → `max_length = 128`

>  NSMC는 짧은 리뷰가 대부분이라 보통 **64로 충분**.  
> max(140자)에 맞추면 95%가 [PAD]로 도배돼서 GPU 낭비.

### 결정 2. Bucketing 효과 예상
- `max / median` 비율을 본다
- **3배 이상** → bucketing 효과 큼 (STEP 5에서 속도 개선 기대)
- **2배 미만** → 효과 작음 (이미 길이가 균일)

##  STEP 1 결론

| 항목 | 값 | 근거 |
|------|-----|------|
| 데이터 크기 | train 150K / test 50K | 충분히 큼, 별도 split 불필요 |
| 클래스 균형 | 50:50 | accuracy로 평가 OK |
| 결측 처리 | null 5개 → filter로 제거 | STEP 3에서 처리 |
| **`max_length`** | **128** | 95%값(107자) 커버, 99%까지 안전 |
| **Bucketing 기대 효과** | **🔥 큼 (max/median = 5.4배)** | STEP 5에서 검증 |

> STEP 5에서 같은 max_length로 with/without bucketing 학습 시간을 비교해서  
> "긴 시퀀스 한도(128)에서 bucketing이 얼마나 패딩을 줄여주는지" 정량 확인 예정.

---
# STEP 2. klue/bert-base 모델 및 Tokenizer 불러오기

##  영입할 명장
- **klue/bert-base**: KLUE 벤치마크 팀이 62GB 한국어 코퍼스로 pretrain
- 약 110M 파라미터 (우리 mini BERT의 80배 규모)
- WordPiece tokenizer, vocab 32,000

## 무엇을 하나
1. `AutoTokenizer.from_pretrained` — 같은 vocab을 쓰는 tokenizer
2. `AutoModelForSequenceClassification` — BERT 본체 + 분류 head 자동 부착
3. 정상 로드되는지, 토크나이저가 한국어를 잘 자르는지 확인

> ⚠️ 분류 head는 **랜덤 초기화** 상태로 부착됨. 학습으로 채워넣는 게 STEP 3~4의 일.

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "klue/bert-base"

# ── 토크나이저: 텍스트 → 토큰 ID 변환기 ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ── 모델: BERT 본체 + 분류 head (2-class) ──
#    id2label/label2id는 결과 해석할 때 편하라고 붙이는 메타 정보
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)

# ── 정상 로드 확인 ──
print(f"Tokenizer vocab size : {tokenizer.vocab_size:,}")
print(f"Model parameters     : {model.num_parameters():,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer vocab size : 32,000
Model parameters     : 110,618,882


In [8]:
# 한국어 문장 하나로 토크나이저가 어떻게 자르는지 직접 확인
sample = "홍명보는 경질되어야 합니다 ㅠㅠ"

enc = tokenizer(sample, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

print(f"원문      : {sample}")
print(f"토큰      : {tokens}")
print(f"input_ids : {enc['input_ids'][0].tolist()}")
print(f"길이      : {len(tokens)} 토큰")

원문      : 홍명보는 경질되어야 합니다 ㅠㅠ
토큰      : ['[CLS]', '홍명보', '##는', '경질', '##되', '##어야', '합니다', 'ㅠㅠ', '[SEP]']
input_ids : [2, 20912, 2259, 18659, 2496, 13091, 3803, 6516, 3]
길이      : 9 토큰


## 3-1. 결측·빈 문자열 제거

STEP 1에서 발견한 null 5개를 제거. 안 하면 tokenizer가 `None`을 만나 에러.

> `dataset.filter()` 는 조건 True인 샘플만 남김. pandas의 `df[조건]`이랑 비슷.

In [9]:
# 결측/빈 문자열 필터링
def is_valid(example):
    doc = example["document"]
    return doc is not None and len(str(doc).strip()) > 0

clean_datasets = raw_datasets.filter(is_valid)

# 필터 전후 크기 비교
print("Before filter:", {k: len(v) for k, v in raw_datasets.items()})
print("After  filter:", {k: len(v) for k, v in clean_datasets.items()})

Filter:   0%|          | 0/150000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

Before filter: {'train': 150000, 'test': 50000}
After  filter: {'train': 149995, 'test': 49997}


## 3-2. 토큰화 — 텍스트를 모델이 먹을 수 있는 숫자로

STEP 1 결정: **`max_length = 128`** (95%값 107자 + 99%까지 안전 커버)

> ⚠️ 여기서 `padding=False`로 두는 게 포인트.  
> 지금 패딩하면 모든 문장이 128 길이로 통일됨 → 메모리 낭비, bucketing 불가.  
> 패딩은 **배치 만들 때 동적으로** (= Collator의 역할, 다음 셀).

In [10]:
MAX_LEN = 128  # STEP 1 분석 결과 기반

def tokenize_fn(batch):
    """
    배치 단위 토큰화 함수.
    truncation=True: 128 토큰 넘으면 자름
    padding=False  : 여기서는 패딩 X (Collator가 동적으로 처리)
    """
    return tokenizer(
        batch["document"],
        truncation=True,
        max_length=MAX_LEN,
    )

tokenized = clean_datasets.map(
    tokenize_fn,
    batched=True,                          # 배치 단위 처리 → 빠름
    remove_columns=["id", "document"],     # 토큰화 후 원본 텍스트는 불필요
)

print(tokenized)
print("\n첫 샘플:", tokenized["train"][0])

Map:   0%|          | 0/149995 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 149995
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 49997
    })
})

첫 샘플: {'label': 0, 'input_ids': [2, 1376, 831, 2604, 18, 18, 4229, 9801, 2075, 2203, 2182, 4243, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## 3-3. Data Collator — 배치 단위 동적 패딩

### 🔧 Collator가 뭐 하는 놈?
"여러 샘플을 모아 **한 배치(텐서)**로 만드는 함수." 그런데 NLP에선 샘플 길이가 다 다르니 패딩을 어떻게 할지가 문제. 

### Static vs Dynamic Padding

| 방식 | 패딩 기준 | 효율 |
|------|---------|------|
| Static (전체 max_length) | 모든 배치 128 토큰 | 낭비 큼 |
| **Dynamic (배치 내 최대 길이)** | 배치마다 다름 | **효율 ↑** ✅ |

> 💡 예시: 한 배치에 짧은 문장만 들어오면 그 배치는 30 토큰까지만 패딩.  
> 이게 **STEP 5 Bucketing의 사전 조건**. (비슷한 길이끼리 모으면 패딩이 더 줄어듦)

In [11]:
from transformers import DataCollatorWithPadding

# 동적 패딩: 배치 안에서 가장 긴 시퀀스 길이에만 맞춤
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 동작 확인: 길이가 다른 샘플 3개를 묶어보기
sample_batch = [tokenized["train"][i] for i in range(3)]
batched = data_collator(sample_batch)

print("배치 모양:")
for k, v in batched.items():
    print(f"  {k:20s}: shape={tuple(v.shape)}")
print(f"\n→ 이 배치는 {batched['input_ids'].shape[1]} 토큰으로 패딩됨 (128 아님!)")

배치 모양:
  input_ids           : shape=(3, 25)
  token_type_ids      : shape=(3, 25)
  attention_mask      : shape=(3, 25)
  labels              : shape=(3,)

→ 이 배치는 25 토큰으로 패딩됨 (128 아님!)


## 3-4. Baseline 학습 — 일단 돌려보기

### 평가 함수
STEP 1에서 라벨 50:50 확인했으니 **accuracy**로 충분.

### TrainingArguments 설정 (Baseline)
- `num_train_epochs=1` ← 빠르게 한 바퀴, 정상 동작 확인 목적
- `learning_rate=5e-5` ← 흔한 기본값 (STEP 4에서 조정)
- `fp16=True` ← GPU 있으면 메모리·속도 ↑
- 저장/체크포인트는 baseline이라 생략



In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """Trainer가 매 eval마다 호출. logits/labels 받아 dict 반환."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds),
    }

In [13]:
import os, time
from transformers import TrainingArguments, Trainer

os.environ["WANDB_DISABLED"] = "true"  # wandb 자동 활성화 방지

baseline_args = TrainingArguments(
    output_dir="./outputs/baseline",
    num_train_epochs=1,                     # baseline: 1 epoch
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,                     # 흔한 기본값
    eval_strategy="epoch",                  # epoch 끝날 때 평가
    logging_steps=200,
    save_strategy="no",                     # baseline은 저장 안 함
    report_to="none",
    fp16=True,                              # GPU 가속
    seed=42,
)

baseline_trainer = Trainer(
    model=model,
    args=baseline_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 학습 시간 측정 (STEP 5에서 bucketing과 비교할 때 필요)
t0 = time.time()
baseline_trainer.train()
baseline_time = time.time() - t0

baseline_eval = baseline_trainer.evaluate()

print(f"\n  Baseline 학습 시간: {baseline_time:.1f}s")
print(f" Baseline 평가     : {baseline_eval}")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.216460,0.236494,0.904174,0.905292


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.216460,0.236494,1,0.904174,0.905292



  Baseline 학습 시간: 697.3s
 Baseline 평가     : {'eval_loss': 0.2364937961101532, 'eval_accuracy': 0.9041742504550273, 'eval_f1': 0.9052918734062111}


In [1]:
import torch
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device         :", torch.cuda.get_device_name(0))
    print("VRAM (GB)      :", torch.cuda.get_device_properties(0).total_memory / 1e9)

CUDA available : True
Device         : Tesla T4
VRAM (GB)      : 15.655829504


 학습 시간 : 697.3s (≈ 11분 37초)\
 eval_accuracy: 0.9042  ← 🤯 1 epoch만에 90% 넘김!\
 eval_f1      : 0.9053\
 eval_loss    : 0.2365\
 GPU          : Tesla T4 (15.6GB VRAM)

"baseline 90.4%에서 +1%p 이상 끌어올리고, 그 개선이 어디서 왔는지 설명할 수 있게 하기."

1 epoch에서 2 epoch로 늘리면 더 잘 학습되겠지만, overfit 위험도 같이 올라가. 그래서:\
\
lr 낮추기(5e-5 → 2e-5) = 한 step씩 조심스럽게\
warmup = 시작할 때 더 조심\
weight decay = 너무 큰 가중치에 패널티\
best model 자동 저장 = overfit 시작 전 시점으로 되돌리기

---
# STEP 4. Fine-tuning으로 성능 더 끌어올리기

## Baseline 결과 회고
- 시간: 697s / accuracy: **0.9042** / f1: 0.9053
-  1 epoch만에 이미 루브릭 90% 달성 → fine-tuning의 위력 확인

## 그래도 더 끌어올리는 이유
1. 의미 있는 개선폭 확보 (90.4% → 91.5%+)
2. STEP 5 bucketing 비교 시 더 정밀한 기준선 필요
3. 학습 곡선 관찰 → 회고에 쓸 분석 자료 확보

## 🛠️ 변경 사항 4종 세트

| 항목 | Baseline | Improved | 의도 |
|------|---------|----------|------|
| Epoch | 1 | **2** | 더 학습 |
| Learning rate | 5e-5 | **2e-5** | 더 조심스럽게 |
| Warmup | ❌ | **10%** | 초반 안정화 |
| Weight decay | 0 | **0.01** | overfit 억제 |
| Best model | ❌ | **자동 복원** | 최적 시점 사용 |

> 결론: "더 깊게 학습 + 안전장치 4중 채우기" 홍명보호와의 차이를 더 벌려보자!!

In [14]:
# Baseline 학습한 모델을 그대로 쓰면 이미 90% 시작점 → 비교가 흐려짐.
# 같은 시작점에서 다시 학습해야 "개선 효과"가 깔끔히 측정됨.

from transformers import AutoModelForSequenceClassification

model_v2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
).to("cuda")   # GPU로 명시적 이동

print("Model v2 device:", next(model_v2.parameters()).device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model v2 device: cuda:0


In [15]:
improved_args = TrainingArguments(
    output_dir="./outputs/improved",
    
    # ── 학습 분량 ──
    num_train_epochs=2,                    # 1 → 2
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    
    # ── Optimizer 튜닝 ──
    learning_rate=2e-5,                    # 5e-5 → 2e-5 (BERT 권장)
    weight_decay=0.01,                     # overfit 억제
    warmup_ratio=0.1,                      # 초반 10% 워밍업
    
    # ── 평가 & 저장 ──
    eval_strategy="steps",                 # epoch → steps
    eval_steps=500,                        # 500 step마다 평가
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,                    # 최신 2개만 보관 (디스크 절약)
    load_best_model_at_end=True,           # ✨ 학습 끝나면 best step 모델로 복원
    metric_for_best_model="accuracy",
    greater_is_better=True,
    
    # ── 기타 ──
    logging_steps=200,
    report_to="none",
    fp16=True,                             # T4면 효과 큼
    seed=42,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [17]:
import time

improved_trainer = Trainer(
    model=model_v2,
    args=improved_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,            # ← v4.46+ 형식
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

t0 = time.time()
improved_trainer.train()
improved_time = time.time() - t0
improved_eval = improved_trainer.evaluate()

print(f"\n{'='*50}")
print(f"  학습 시간 : {improved_time:.1f}s")
print(f" 평가     : {improved_eval}")
print(f" 개선폭    : Baseline 0.9042 → Improved {improved_eval['eval_accuracy']:.4f}")
print(f"           (+{(improved_eval['eval_accuracy'] - 0.9042)*100:.2f}%p)")

Step,Training Loss,Validation Loss,Accuracy,F1
500,0.386134,0.345034,0.850171,0.858433
1000,0.313256,0.318667,0.870912,0.874874
1500,0.294765,0.282730,0.883733,0.885767
2000,0.264516,0.273384,0.887953,0.886120
2500,0.275726,0.259300,0.892214,0.893848
3000,0.260783,0.271610,0.894014,0.892891
3500,0.246663,0.257343,0.895254,0.893142
4000,0.263536,0.248019,0.896454,0.893466
4500,0.246876,0.248110,0.899634,0.902718
5000,0.187117,0.261317,0.902294,0.903696


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Accuracy,F1
0.177017,0.249615,9376,0.907594,0.908598



  학습 시간 : 2402.8s
 평가     : {'eval_loss': 0.24961452186107635, 'eval_accuracy': 0.9075944556673401, 'eval_f1': 0.9085981086535038}
 개선폭    : Baseline 0.9042 → Improved 0.9076
           (+0.34%p)


---
# STEP 5. Bucketing 적용 → STEP 4와 비교

##  Bucketing이 노리는 것
비슷한 길이끼리 한 배치에 모아서 **[PAD] 연산 낭비를 줄이기**.

- NSMC: median 27자 vs max 146자 → **5.4배 격차**
- max_length 128에서 짧은 문장은 [PAD] 폭격 → bucketing 잠재력 ↑

## 비교 설계 (통제 실험)
STEP 4와 **다른 조건은 모두 동일**, `group_by_length=True`만 추가.

| 항목 | STEP 4 | STEP 5 |
|------|--------|--------|
| Epoch | 2 | 2 |
| LR / Warmup / WD | 동일 | 동일 |
| **group_by_length** | False | **True** ✨ |

## 측정 포인트
1.  **학습 시간** — 줄어드는가? 얼마나?
2.  **Accuracy** — 떨어지는가? trade-off는 어떤가?
3.  **분석** — 왜 그런 결과가 나왔나?

In [18]:
# STEP 4와 같은 출발점에서 시작해야 공정한 비교
from transformers import AutoModelForSequenceClassification

model_v3 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
).to("cuda")

print("Model v3 device:", next(model_v3.parameters()).device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model v3 device: cuda:0


In [25]:
from transformers import AutoModelForSequenceClassification

model_v3 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
).to("cuda")

print("Model v3 device:", next(model_v3.parameters()).device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model v3 device: cuda:0


In [26]:
def add_length(example):
    example["length"] = len(example["input_ids"])
    return example

tokenized_with_length = tokenized.map(add_length)
print(tokenized_with_length)
print("\n첫 샘플 길이:", tokenized_with_length["train"][0]["length"])

Map:   0%|          | 0/149995 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask', 'length'],
        num_rows: 149995
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask', 'length'],
        num_rows: 49997
    })
})

첫 샘플 길이: 13


In [27]:
from transformers import Trainer
from transformers.trainer_pt_utils import LengthGroupedSampler


class BucketingTrainer(Trainer):
   
    
    def _get_train_sampler(self, *args, **kwargs):
        lengths = self.train_dataset["length"]
        return LengthGroupedSampler(
            batch_size=self.args.train_batch_size,
            dataset=self.train_dataset,
            lengths=lengths,
            model_input_name="input_ids",
        )

In [28]:
bucketing_args = TrainingArguments(
    output_dir="./outputs/bucketing",
    
    # ── STEP 4와 완전히 동일하게 유지 ──
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=200,
    report_to="none",
    fp16=True,
    seed=42,
    # group_by_length 없음 — sampler 주입으로 대체
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [29]:
bucketing_trainer = BucketingTrainer(    # ← Trainer 대신 BucketingTrainer
    model=model_v3,
    args=bucketing_args,
    train_dataset=tokenized_with_length["train"],
    eval_dataset=tokenized_with_length["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

t0 = time.time()
bucketing_trainer.train()
bucketing_time = time.time() - t0
bucketing_eval = bucketing_trainer.evaluate()

print(f"\n{'='*50}")
print(f"  Bucketing 학습 시간 : {bucketing_time:.1f}s")
print(f" Bucketing 평가     : {bucketing_eval}")

Step,Training Loss,Validation Loss,Accuracy,F1
500,0.388302,0.339506,0.859312,0.853464
1000,0.310990,0.319229,0.876313,0.876290
1500,0.299258,0.282636,0.882713,0.882064
2000,0.274627,0.267410,0.887093,0.888269
2500,0.270429,0.261135,0.891974,0.893955
3000,0.253806,0.257391,0.894194,0.892471
3500,0.266872,0.254171,0.895834,0.893527
4000,0.240813,0.258000,0.894234,0.897805
4500,0.243624,0.246938,0.898494,0.902142
5000,0.186256,0.259484,0.901934,0.902763


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Accuracy,F1
0.170737,0.246276,9376,0.906374,0.907871



  Bucketing 학습 시간 : 2033.4s
 Bucketing 평가     : {'eval_loss': 0.24627593159675598, 'eval_accuracy': 0.9063743824629478, 'eval_f1': 0.9078706528370958}


---
#  최종 회고 및 결론

## 1. 루브릭 달성 요약

| # | 학습목표 | 평가기준 | 결과 |
|---|---------|---------|------|
| 1 | 모델·데이터 정상 로드 및 작동 | klue/bert-base를 NSMC로 fine-tuning, 정상 동작 | ✅ STEP 3 baseline 학습 완료 |
| 2 | Preprocessing 개선 + Fine-tuning으로 성능 개선 | Validation accuracy **≥ 90%** | ✅ **0.9076** (STEP 4) |
| 3 | Bucketing 성공 적용 + 결과 비교분석 | 연산속도 vs 모델성능 trade-off 분석 | ✅ **시간 -15.4% / Acc -0.12%p** |

## 2. 정량 결과 요약

| Stage | Epoch | 시간 | Accuracy | F1 | 비고 |
|-------|-------|------|----------|-----|------|
| Baseline (STEP 3) | 1 | 697s | 0.9042 | 0.9053 | 기본 lr=5e-5 |
| Improved (STEP 4) | 2 | 2,403s | **0.9076** | 0.9086 | lr=2e-5 + warmup + decay + best 복원 |
| Bucketing (STEP 5) | 2 | **2,033s** | 0.9064 | 0.9079 | LengthGroupedSampler 주입 |

## 3. 핵심 발견 & 분석

###  발견 1: Fine-tuning은 1 epoch면 충분 (NSMC + klue/bert-base 한정)

Baseline(1 epoch)이 이미 90.4%를 달성. STEP 4에서 2 epoch + 안전장치 4종(lr↓, warmup, weight decay, best model 복원)을 추가했지만 개선폭은 **+0.34%p**에 그침. 학습 곡선상 Epoch 2 진입 시점에서 train loss는 0.25→0.18로 급감하는데 **val loss는 오히려 0.248→0.261로 상승** → overfit 시작 신호 관측. **klue/bert-base의 pretrain이 워낙 잘 되어있어 추가 학습의 한계 효용이 작음**.

###  발견 2: Bucketing은 NSMC에서 사실상 win-win

| 가설 | 실측 |
|------|------|
| 시간 30~45% 절감 예상 | **15.4% 절감** |
| Acc 0.1~0.5%p 감소 예상 | **0.12%p 감소** |

기대만큼 시간이 줄지 않은이유:
- **fp16 + T4 GPU 환경에서는 패딩 연산이 이미 빨라** 절감 여지가 작음
- GPU는 행렬 곱 효율이 높아 짧은 시퀀스의 상대적 이점이 줄어듦

→ **CPU나 더 작은 GPU에서는 절감 폭이 훨씬 클 것으로 추정**.

###  발견 3: 분류 task에서 bucketing 성능 손실이 작은 이유

BERT 분류는 **[CLS] 토큰 하나의 벡터로 결정**되므로 배치 내 길이 다양성 영향이 작음. 만약 token-level task(NER)나 생성 task였다면 성능 차이가 더 컸을 것으로 추정.

## 4. 구현 디테일 — 회고에서 짚을 만한 것

1. **`group_by_length` 인자가 transformers 5.9.0에서 제거됨** → `LengthGroupedSampler`를 직접 주입하는 `BucketingTrainer` 클래스 작성. 결과적으로 bucketing의 동작 메커니즘(mega-batch 길이 정렬 → batch 셔플)을 코드 레벨에서 통제하는 경험.
2. **동적 패딩(`DataCollatorWithPadding`)** — `max_length`로 일괄 패딩이 아닌 배치 단위 동적 패딩이 bucketing의 사전 조건. 두 기법이 짝으로 동작.
3. **`max_length=128` 결정의 근거**: STEP 1에서 글자 길이 95% = 107자 측정 → 한국어 WordPiece 팽창률 1.2배 고려 → 128 토큰 결정. **데이터 분석이 하이퍼파라미터 결정의 근거가 된 좋은 사례**.

## 5. 배운 점

###  개념 차원
- **Fine-tuning은 "셰프 영입 + 메뉴판 갈아끼우기"**: classifier head 외엔 전부 pretrain 가중치 재사용. 110M 파라미터 중 새로 배우는 건 단 ~1,500개(768×2+2).
- **데이터를 보는 것이 모델을 보는 것보다 먼저**: STEP 1의 길이 분포 분석이 STEP 3의 `max_length`, STEP 5의 bucketing 효과 예측까지 일관되게 연결됨.
- **Trade-off 분석의 본질**: "공짜 점심은 없다"가 아니라 "어떤 점심은 거의 공짜"임을 정량적으로 측정해야 알 수 있음.

###  엔지니어링 차원
- HF Trainer API의 인자 변경에 대응하는 법 (`tokenizer` → `processing_class`, `group_by_length` → sampler 주입)
- 통제 실험 설계: STEP 4·5의 비교에서 **bucketing 외 모든 조건 동일하게 유지**
- GPU 환경 진단(`torch.cuda.is_available`, `nvidia-smi`)과 `.to("cuda")` 명시적 디바이스 이동

## 6. 한계 & 개선 방향

| 항목 | 현재 | 개선 방향 |
|------|------|----------|
| 효과 측정 환경 | T4 GPU + fp16 | CPU 또는 V100에서도 측정해 환경별 효과 비교 |
| Bucketing 변형 | 기본 LengthGroupedSampler | mega_batch_mult 조정으로 정렬 강도 실험 |
| 비교 baseline | 2 epoch 단일 시도 | seed 3~5개 평균으로 분산 통제 |
| Downstream task | NSMC 분류만 | NER, 생성 task에서도 bucketing 효과 측정 |

## 7. 마치며

> 지난 mini BERT pretrain에서 **MLM/NSP로 한국어 표현을 직접 학습**시키는 과정을 경험했다면, 이번 프로젝트는 그 표현을 **specific task(감정 분류)에 맞추는 fine-tuning** 단계를 경험한 셈.
>
> 가장 인상 깊었던 것은 STEP 3의 baseline 1 epoch만으로 **90.4%가 나온 순간**. KLUE 팀이 62GB로 미리 학습해둔 가중치 덕분에, 우리는 1,500개의 새 파라미터(classifier head)와 2 epoch의 fine-tuning만으로 production-grade 분류기를 만들 수 있었다. **거대 모델 시대의 작업 방식 = "from scratch가 아니라 from somewhere"** 임을 체감.
>
> Bucketing 실험에서는 "시간 -15%, 성능 -0.12%p"라는 비대칭 trade-off를 관측. 단순히 "효과 있다/없다"가 아니라 **하드웨어·task 특성에 따라 효과가 달라진다는 메타-통찰**을 얻은 게 더 큰 수확.
>
> 다음에는 같은 데이터로 RoBERTa나 ELECTRA를 fine-tuning해서 모델 아키텍처별 성능 차이를 보거나, KorNLI 같은 더 어려운 task로 fine-tuning의 한계를 탐구해보고 싶다.